In [1]:
import sys
import os
import time
import math
import torch

SRC_PATH = os.path.normpath(os.path.join(os.path.abspath(''), '..', 'src'))
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

from models.cpmp_transformer_v9 import CPMPTransformer
from training.training import load_model, load_hyperparams
from solvers.model import ModelSolver
from settings import INSTANCE_FOLDER, MODELS_FOLDER

In [2]:
# --- Modelo SL (v9_dataBSG_250k) ---
model_sl = load_model(CPMPTransformer, 'v9_dataBSG_250k')
print(f'Modelo SL cargado — parámetros: {sum(p.numel() for p in model_sl.parameters()):,}')

# --- Modelo RL (checkpoint REINFORCE) ---
hyperparams = load_hyperparams('v9_dataBSG_250k')  # misma arquitectura
model_rl = CPMPTransformer(**hyperparams)
ckpt = torch.load(str(MODELS_FOLDER / 'v9_rl_checkpoint.pth'), map_location='cpu', weights_only=False)
model_rl.load_state_dict(ckpt['model_state_dict'])
model_rl.eval()
print(f'Modelo RL cargado — epoch {ckpt["epoch"]}, step {ckpt["step"]}')

class SafeModelSolver(ModelSolver):
    def solve_from_path(self, instance_path, H, max_steps):
        try:
            return super().solve_from_path(instance_path, H, max_steps)
        except (ValueError, RuntimeError, IndexError):
            return False, max_steps

solver_sl = SafeModelSolver(model_sl)
solver_rl = SafeModelSolver(model_rl)
print(f'\nSolvers listos: SL vs RL')

Modelo SL cargado — parámetros: 285,184
Modelo RL cargado — epoch 50, step 31400

Solvers listos: SL vs RL


In [3]:
def benchmark_solver(solver, dat_files, H, max_steps):
    solved_list, steps_list, time_list = [], [], []
    for filepath in dat_files:
        t0 = time.perf_counter()
        solved, steps = solver.solve_from_path(filepath, H, max_steps)
        elapsed = time.perf_counter() - t0
        solved_list.append(solved)
        steps_list.append(steps)
        time_list.append(elapsed)
    return solved_list, steps_list, time_list

def avg(values, mask=None):
    if mask is not None:
        vals = [v for v, ok in zip(values, mask) if ok]
    else:
        vals = list(values)
    return sum(vals) / len(vals) if vals else math.nan

def pct_better_quality(avg_sl, avg_rl):
    if math.isnan(avg_sl) or math.isnan(avg_rl):
        return None, math.nan
    if avg_sl < avg_rl:
        return 'SL', (avg_rl - avg_sl) / avg_rl * 100
    elif avg_rl < avg_sl:
        return 'RL', (avg_sl - avg_rl) / avg_sl * 100
    return 'Empate', 0.0

In [4]:
CVS_PATH = INSTANCE_FOLDER / 'benchmarks' / 'CVS'
MAX_STEPS = 100

cvs_folders = sorted(
    [d for d in os.listdir(CVS_PATH) if (CVS_PATH / d).is_dir()]
)

results = {}

print(f'Corriendo benchmark en {len(cvs_folders)} categorías CVS...\n')

for folder_name in cvs_folders:
    H_file, S = [int(x) for x in folder_name.split('-')]
    H = H_file + 2
    folder_path = CVS_PATH / folder_name

    dat_files = sorted([
        str(folder_path / f)
        for f in os.listdir(folder_path)
        if f.endswith('.dat')
    ])

    print(f'  [{folder_name}] {len(dat_files)} instancias — H_archivo={H_file}, H_real={H}, S={S}')

    solved_sl, steps_sl, time_sl = benchmark_solver(solver_sl, dat_files, H, MAX_STEPS)
    solved_rl, steps_rl, time_rl = benchmark_solver(solver_rl, dat_files, H, MAX_STEPS)

    results[folder_name] = {
        'H_file': H_file, 'H': H, 'S': S, 'n': len(dat_files),
        'solved_sl': solved_sl, 'steps_sl': steps_sl, 'time_sl': time_sl,
        'solved_rl': solved_rl, 'steps_rl': steps_rl, 'time_rl': time_rl,
    }

print('\nBenchmark completado.')

Corriendo benchmark en 21 categorías CVS...

  [10-10] 40 instancias — H_archivo=10, H_real=12, S=10


/mnt/data/Proyectos/Universidad/CPMP-Transformer/.venv/lib/python3.14/site-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


  [10-6] 40 instancias — H_archivo=10, H_real=12, S=6
  [3-3] 40 instancias — H_archivo=3, H_real=5, S=3
  [3-4] 40 instancias — H_archivo=3, H_real=5, S=4
  [3-5] 40 instancias — H_archivo=3, H_real=5, S=5
  [3-6] 40 instancias — H_archivo=3, H_real=5, S=6
  [3-7] 40 instancias — H_archivo=3, H_real=5, S=7
  [3-8] 40 instancias — H_archivo=3, H_real=5, S=8
  [4-4] 40 instancias — H_archivo=4, H_real=6, S=4
  [4-5] 40 instancias — H_archivo=4, H_real=6, S=5
  [4-6] 40 instancias — H_archivo=4, H_real=6, S=6
  [4-7] 40 instancias — H_archivo=4, H_real=6, S=7
  [5-10] 40 instancias — H_archivo=5, H_real=7, S=10
  [5-4] 40 instancias — H_archivo=5, H_real=7, S=4
  [5-5] 40 instancias — H_archivo=5, H_real=7, S=5
  [5-6] 40 instancias — H_archivo=5, H_real=7, S=6
  [5-7] 40 instancias — H_archivo=5, H_real=7, S=7
  [5-8] 40 instancias — H_archivo=5, H_real=7, S=8
  [5-9] 40 instancias — H_archivo=5, H_real=7, S=9
  [6-10] 40 instancias — H_archivo=6, H_real=8, S=10
  [6-6] 40 instancias — 

In [5]:
W = 150
HDR_SEP = '═' * W

col_hdr = (
    f"{'Categoría':>10}  {'H':>3} {'S':>3} {'N':>4}  "
    f"{'─── SL (v9_dataBSG_250k) ───':^38}  "
    f"{'─── RL (REINFORCE) ───':^38}  "
    f"{'Calidad (pasos)':^20}"
)
sub_hdr = (
    f"{'':>10}  {'':>3} {'':>3} {'':>4}  "
    f"{'resuelto':>10} {'avg pasos':>10} {'avg tiempo':>12}  "
    f"{'resuelto':>10} {'avg pasos':>10} {'avg tiempo':>12}  "
    f"{'ganador':^20}"
)

print(HDR_SEP)
print(col_hdr)
print(sub_hdr)
print(HDR_SEP)

for folder_name in cvs_folders:
    r = results[folder_name]
    n = r['n']
    n_sl = sum(r['solved_sl'])
    n_rl = sum(r['solved_rl'])

    both = [a and b for a, b in zip(r['solved_sl'], r['solved_rl'])]

    avg_steps_sl      = avg(r['steps_sl'], r['solved_sl'])
    avg_steps_rl      = avg(r['steps_rl'], r['solved_rl'])
    avg_steps_sl_both = avg(r['steps_sl'], both)
    avg_steps_rl_both = avg(r['steps_rl'], both)
    avg_time_sl       = avg(r['time_sl'])
    avg_time_rl       = avg(r['time_rl'])

    winner, pct = pct_better_quality(avg_steps_sl_both, avg_steps_rl_both)

    def fmt_steps(a): return f'{a:>8.1f}' if not math.isnan(a) else '       -'
    def fmt_time(a):  return f'{a:>10.4f}s' if not math.isnan(a) else '          -'
    def fmt_winner(w, p):
        if w is None or math.isnan(p): return '          -'
        if w == 'Empate': return '      Empate'
        return f'{w} +{p:5.1f}%'

    print(
        f"{folder_name:>10}  {r['H_file']:>3} {r['S']:>3} {n:>4}  "
        f"{n_sl:>3}/{n:<3} ({n_sl/n:>5.1%}) {fmt_steps(avg_steps_sl)} {fmt_time(avg_time_sl)}  "
        f"{n_rl:>3}/{n:<3} ({n_rl/n:>5.1%}) {fmt_steps(avg_steps_rl)} {fmt_time(avg_time_rl)}  "
        f"{fmt_winner(winner, pct):^20}"
    )

print(HDR_SEP)

══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
 Categoría    H   S    N       ─── SL (v9_dataBSG_250k) ───               ─── RL (REINFORCE) ───            Calidad (pasos)   
                            resuelto  avg pasos   avg tiempo    resuelto  avg pasos   avg tiempo        ganador       
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
     10-10   10  10   40    0/40  ( 0.0%)        -     0.1535s    0/40  ( 0.0%)        -     0.1519s                -     
      10-6   10   6   40    0/40  ( 0.0%)        -     0.1378s    0/40  ( 0.0%)        -     0.1352s                -     
       3-3    3   3   40   40/40  (100.0%)     10.2     0.0124s   40/40  (100.0%)     10.4     0.0123s       SL +  1.9%     
       3-4    3   4   40   40/40  (100.0%)     10.9     0.0133s   40/40  (100.0%)

In [6]:
# Separar H<10 y H=10 para el resumen
def global_summary(label, folder_filter):
    sel = {k: v for k, v in results.items() if folder_filter(v)}
    if not sel:
        return

    all_solved_sl = [s for r in sel.values() for s in r['solved_sl']]
    all_steps_sl  = [s for r in sel.values() for s in r['steps_sl']]
    all_solved_rl = [s for r in sel.values() for s in r['solved_rl']]
    all_steps_rl  = [s for r in sel.values() for s in r['steps_rl']]

    total_n  = len(all_solved_sl)
    total_sl = sum(all_solved_sl)
    total_rl = sum(all_solved_rl)

    both = [a and b for a, b in zip(all_solved_sl, all_solved_rl)]
    avg_sl      = avg(all_steps_sl, all_solved_sl)
    avg_rl      = avg(all_steps_rl, all_solved_rl)
    avg_sl_both = avg(all_steps_sl, both)
    avg_rl_both = avg(all_steps_rl, both)

    winner, pct = pct_better_quality(avg_sl_both, avg_rl_both)
    winner_str = f'{winner} +{pct:.1f}%' if winner not in (None, 'Empate') else (winner or '-')

    BW = 56
    print('═' * BW)
    print(f"  RESUMEN — {label}")
    print('═' * BW)
    print(f"  Instancias           : {total_n}")
    print(f"  SL resueltas         : {total_sl}/{total_n} ({total_sl/total_n:.2%})")
    print(f"  RL resueltas         : {total_rl}/{total_n} ({total_rl/total_n:.2%})")
    diff = total_rl - total_sl
    print(f"  Diferencia           : {'+'if diff>=0 else ''}{diff} instancias a favor {'RL' if diff>=0 else 'SL'}")
    print()
    print(f"  Avg pasos (propios resueltos)")
    print(f"    SL : {avg_sl:.2f} pasos")
    print(f"    RL : {avg_rl:.2f} pasos")
    print(f"  Avg pasos (resueltos por ambos)")
    print(f"    SL : {avg_sl_both:.2f} pasos")
    print(f"    RL : {avg_rl_both:.2f} pasos")
    print(f"  Calidad              : {winner_str}")
    print('═' * BW)
    print()

global_summary('H < 10 (instancias relevantes)', lambda r: r['H_file'] < 10)
global_summary('H = 10 (instancias difíciles)',   lambda r: r['H_file'] == 10)
global_summary('TOTAL (840 instancias)',           lambda r: True)

════════════════════════════════════════════════════════
  RESUMEN — H < 10 (instancias relevantes)
════════════════════════════════════════════════════════
  Instancias           : 760
  SL resueltas         : 757/760 (99.61%)
  RL resueltas         : 756/760 (99.47%)
  Diferencia           : -1 instancias a favor SL

  Avg pasos (propios resueltos)
    SL : 30.42 pasos
    RL : 29.81 pasos
  Avg pasos (resueltos por ambos)
    SL : 30.31 pasos
    RL : 29.69 pasos
  Calidad              : RL +2.1%
════════════════════════════════════════════════════════

════════════════════════════════════════════════════════
  RESUMEN — H = 10 (instancias difíciles)
════════════════════════════════════════════════════════
  Instancias           : 80
  SL resueltas         : 0/80 (0.00%)
  RL resueltas         : 0/80 (0.00%)
  Diferencia           : +0 instancias a favor RL

  Avg pasos (propios resueltos)
    SL : nan pasos
    RL : nan pasos
  Avg pasos (resueltos por ambos)
    SL : nan pasos
   